# MVP04 — Camada Silver

Lê a camada `bronze`, diagnostica a qualidade dos dados, aplica os tratamentos
necessários e persiste o resultado na camada `silver`.

**Posição no pipeline:** quarto notebook. Requer `MVP03-Bronze` executado.

**Princípio da camada:** a Silver mantém a granularidade da origem, mas entrega o dado
limpo, tipado e confiável. É aqui que os problemas de qualidade são identificados e
resolvidos — nenhuma camada posterior precisa tratá-los de novo.

## Estrutura do notebook

| Etapa | Conteúdo |
|---|---|
| 1 | Parâmetros e funções auxiliares |
| 2 | **Diagnóstico de qualidade sobre a Bronze** — antes de qualquer tratamento |
| 3 | Transformações e correções, tabela por tabela |
| 4 | **Validação pós-tratamento** — os mesmos testes, agora sobre a Silver |
| 5 | Documentação no Unity Catalog |
| 6 | Resumo do tratamento aplicado |

Rodar os mesmos testes nas etapas 2 e 4 evidencia tanto o problema encontrado quanto a sua resolução.

## 📚 Sobre este Notebook

Este notebook implementa a ingestão da **camada Silver** a partir dos dados já carregados na **camada Bronze**.

### Arquitetura Medalhão

* **📂 Staging**: Dados brutos copiados do Google Drive (notebook MVP02-Download)
* **🥉 Bronze**: Dados brutos com metadados de rastreabilidade (notebook MVP03-Bronze)
* **🥈 Silver**: Dados limpos, padronizados e com tipos corretos (este notebook)
* **🥇 Gold**: Dados agregados e prontos para consumo  (notebook MVP05-Gold)

### 🎯 Objetivo
Ler as tabelas da camada Bronze e aplicnar as seguintes transformações:

* **Normalização de nomes**: colunas em lowercase com separador `_`
* **Conversão de tipos**: strings de data → `DATE`/`TIMESTAMP`, strings numéricas → `DOUBLE`/`INT`
* **Tratamento de valores especiais**: `-` → `NULL`
* **Decimais**: trocar vírgula por ponto (padrão brasileiro → padrão internacional)
* **Metadados de rastreabilidade**: `data_carga_silver` e `origem_tabela_bronze`

In [0]:
%sql
-- Define o catálogo ativo como mvpDataEngineer no Unity Catalog.
USE CATALOG mvpDataEngineer;

-- Define o schema ativo como silver dentro do catálogo atual.
USE SCHEMA silver;

## 🔧 Etapa 1 — Parâmetros e funções auxiliares

### Colunas descartadas nesta camada

Cinco colunas presentes nas três tabelas de origem **não seguem para a Silver**, por não
terem valor analítico:

| Coluna | Motivo do descarte |
|---|---|
| `PROTOCOLO` | Identificador do formulário de origem. É metadado do sistema de lançamento, não do fenômeno medido. Nenhuma pergunta de negócio depende dele. |
| `SOLICDATA` | Data do lançamento no sistema, não da medição. Irrelevante para a análise. |
| `STATUS` | Situação no fluxo de aprovação. **Usado como filtro antes de ser descartado** — ver abaixo. |
| `UNIDADE` | Constante em todas as linhas das três tabelas. Coluna sem variação não carrega informação. |
| `OBSERVACOES` | Texto livre, vazio em duas das três fontes e preenchido em 4 de 226 linhas na terceira. Volume insuficiente para tratamento estruturado. |

**`PROTOCOLO` e `SOLICDATA` são usados durante o processamento e descartados ao final.**
Eles são necessários para dois tratamentos: `SOLICDATA` serve de critério de desempate na
deduplicação (o lançamento mais recente corrige o anterior), e `PROTOCOLO` permite
identificar registros atribuídos à entidade ou à data errada. Cumprida essa função, saem.

**`STATUS` é aplicado como filtro antes do descarte.** Apenas registros com
`Solicitação Finalizada` — homologados pela operação — seguem adiante. Depois do filtro,
a coluna passa a ter valor único e é removida pelo mesmo motivo de `UNIDADE`.

In [0]:
CATALOGO = "mvpdataengineer"

STATUS_HOMOLOGADO = "Solicitação Finalizada"

print(f"Catálogo: {CATALOGO}")
print(f"Status homologado: {STATUS_HOMOLOGADO}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def texto_para_numero(coluna, tipo="double"):
    """Converte texto da origem em número.

    Trata as duas formas de ausência presentes nos arquivos (hífen e célula vazia)
    e a vírgula como separador decimal.
    """
    limpo = F.trim(F.col(coluna))
    return (F.when(limpo.isin("-", ""), None)
             .otherwise(F.regexp_replace(limpo, ",", ".").cast(tipo)))


def chave_duplicada(df, chaves, medidas, col_protocolo="PROTOCOLO", col_solicdata="SOLICDATA"):
    """Retorna as chaves de negócio com mais de um registro.

    A CHAVE é a entidade medida mais a DATA DO REGISTRO — a data do fato ocorrido em
    campo. `PROTOCOLO` e `SOLICDATA` NÃO fazem parte da chave: eles identificam o
    lançamento no sistema, não a medição. Dois protocolos lançados em dias diferentes
    que descrevem a mesma leitura da mesma data são exatamente a duplicidade a
    encontrar, e incluí-los na chave a esconderia.

    Também não se agrupa pelas medidas. Agrupar por elas acharia apenas linhas
    idênticas e deixaria passar o caso pior: mesma chave com valores divergentes.

    A classificação resultante:
      - REENVIO           : mesma leitura lançada duas vezes. Deduplicação segura.
      - CONFLITO DE VALOR : mesma chave com valores diferentes. Exige decisão explícita.
    """
    agg = [
        F.count("*").alias("lancamentos"),
        F.collect_set(F.col(col_protocolo).cast("string")).alias("protocolos"),
        F.sort_array(F.collect_set(F.col(col_solicdata).cast("string"))).alias("datas_lancamento"),
    ]
    for m in medidas:
        agg.append(F.countDistinct(m).alias(f"distintos_{m}"))
        agg.append(F.collect_set(F.col(m).cast("string")).alias(f"valores_{m}"))

    # greatest() exige ao menos duas colunas; com uma única medida, usa-se a própria.
    cols_distintos = [F.col(f"distintos_{m}") for m in medidas]
    max_distintos = cols_distintos[0] if len(cols_distintos) == 1 else F.greatest(*cols_distintos)

    return (df.groupBy(*chaves).agg(*agg)
              .filter(F.col("lancamentos") > 1)
              .withColumn("classificacao",
                          F.when(max_distintos > 1, F.lit("CONFLITO DE VALOR"))
                           .otherwise(F.lit("REENVIO")))
              .orderBy(F.desc("classificacao"), *chaves))


def resumo_duplicidade(df, chaves, medidas, rotulo):
    """
    Imprime o placar de duplicidade de uma tabela.
    """
    d = chave_duplicada(df, chaves, medidas)
    total_chaves = df.select(*chaves).distinct().count()
    contagem = {r["classificacao"]: r["qtd"] for r in
                d.groupBy("classificacao").agg(F.count("*").alias("qtd")).collect()}
    reenvio = contagem.get("REENVIO", 0)
    conflito = contagem.get("CONFLITO DE VALOR", 0)
    excedentes = df.count() - total_chaves

    print(f"{rotulo}")
    print(f"  Chave de negócio .................: {chaves}")
    print(f"  Registros na tabela ..............: {df.count()}")
    print(f"  Chaves distintas .................: {total_chaves}")
    print(f"  Lançamentos excedentes ...........: {excedentes}")
    print(f"  Chaves com reenvio ...............: {reenvio}")
    print(f"  Chaves com conflito de valor .....: {conflito}")
    return d


def perfil_completude(df, colunas):
    """Percentual de nulos por coluna."""
    total = df.count()
    linhas = []
    for c in colunas:
        nulos = df.filter(F.col(c).isNull()).count()
        linhas.append({
            "coluna": c,
            "total": total,
            "nulos": nulos,
            "pct_nulos": round(100.0 * nulos / total, 2) if total else None,
        })
    return spark.createDataFrame(linhas).orderBy(F.desc("pct_nulos"))


print("Funções auxiliares definidas.")

# 🔍 Etapa 2 — Diagnóstico de qualidade

Verificação executada **sobre a camada Bronze**, antes de qualquer tratamento. Cada
tabela é avaliada nas cinco dimensões de qualidade: completude, consistência, unicidade,
acurácia e presença de outliers.

As células desta etapa são diagnósticas: não alteram nenhum dado.

In [0]:
bronze_pluv  = spark.table(f"{CATALOGO}.bronze.leitura_pluviometros")
bronze_bacia = spark.table(f"{CATALOGO}.bronze.medicao_bacias")
bronze_laje  = spark.table(f"{CATALOGO}.bronze.medicao_laje")

print("Volume na Bronze")
print(f"  leitura_pluviometros : {bronze_pluv.count():>6} registros")
print(f"  medicao_bacias       : {bronze_bacia.count():>6} registros")
print(f"  medicao_laje         : {bronze_laje.count():>6} registros")

In [0]:
# Consistência — domínio dos campos categóricos
# Verifica se os campos de domínio fechado contêm apenas os valores esperados. Variação de grafia em um campo categórico quebra junções e duplica membros de dimensão.

print("STATUS — medicao_bacias")
bronze_bacia.groupBy("STATUS").count().show(truncate=False)

print("STATUS — leitura_pluviometros")
bronze_pluv.groupBy("STATUS").count().show(truncate=False)

print("STATUS — medicao_laje")
bronze_laje.groupBy("STATUS").count().show(truncate=False)

print("UNIDADE — valores distintos por tabela")
for nome, df in [("pluviometros", bronze_pluv), ("bacias", bronze_bacia), ("laje", bronze_laje)]:
    print(f"  {nome}: {[r[0] for r in df.select('UNIDADE').distinct().collect()]}")

In [0]:
# Rótulos das estações pluviométricas
# A salina tem cinco estações. Se aparecerem mais de cinco rótulos, há variação de grafia.
display(
    bronze_pluv.groupBy("ESTACAO")
        .agg(F.count("*").alias("registros"),
             F.min("DATAREGISTRO").alias("primeiro_registro"),
             F.max("DATAREGISTRO").alias("ultimo_registro"))
        .orderBy("ESTACAO")
)

In [0]:
# Identificadores das bacias

display(bronze_bacia.groupBy("BACIA").agg(F.count("*").alias("registros")).orderBy("BACIA"))

In [0]:
# Consistência — representação de valores ausentes
# Nos arquivos de origem a ausência de valor aparece de duas formas: o caractere `-` e a célula vazia. Ambas precisam ser convertidas para `NULL`, e a contagem abaixo dimensiona o problema.

colunas_numericas_bacia = ["AGUADOCE_COLOCADO", "AGUADOCE_RETIRADO",
                           "SALMOURA_COLOCADO", "SALMOURA_RETIRADO", "SALMOURA_DENSIDADE"]

linhas = []
for c in colunas_numericas_bacia:
    linhas.append({
        "tabela": "medicao_bacias",
        "coluna": c,
        "hifen": bronze_bacia.filter(F.trim(F.col(c)) == "-").count(),
        "vazio_ou_nulo": bronze_bacia.filter(F.col(c).isNull() | (F.trim(F.col(c)) == "")).count(),
        "preenchido": bronze_bacia.filter(
            F.col(c).isNotNull() & (F.trim(F.col(c)) != "-") & (F.trim(F.col(c)) != "")).count(),
    })

linhas.append({
    "tabela": "leitura_pluviometros",
    "coluna": "MILIMETROS",
    "hifen": bronze_pluv.filter(F.trim(F.col("MILIMETROS")) == "-").count(),
    "vazio_ou_nulo": bronze_pluv.filter(
        F.col("MILIMETROS").isNull() | (F.trim(F.col("MILIMETROS")) == "")).count(),
    "preenchido": bronze_pluv.filter(
        F.col("MILIMETROS").isNotNull() & (F.trim(F.col("MILIMETROS")) != "-")
        & (F.trim(F.col("MILIMETROS")) != "")).count(),
})

display(spark.createDataFrame(linhas))

## 2.1 Unicidade — mesma medição lançada mais de uma vez

### Definição da chave de negócio

A chave é a **entidade medida mais a data do registro** — a data em que a medição
ocorreu em campo:

| Tabela | Chave de negócio |
|---|---|
| `leitura_pluviometros` | (`ESTACAO`, `DATAREGISTRO`) |
| `medicao_bacias` | (`BACIA`, `DATAREGISTRO`) |
| `medicao_laje` | (`CRISTALIZADOR`, `DATAMEDICAO`) |

**`PROTOCOLO` e `SOLICDATA` ficam fora da chave**, e é isso que faz o teste funcionar.
Eles identificam o *lançamento no sistema*, não a *medição*. O caso típico:

| Protocolo | SOLICDATA | DATAREGISTRO | Milímetros |
|---|---|---|---|
| 215487 | 01/07/2026 | 01/07/2026 | 12,5 |
| 215479 | 02/07/2026 | 01/07/2026 | 12,5 |

São duas inclusões no sistema, em dias diferentes, descrevendo **a mesma leitura de
12,5 mm do dia 01/07**. Existe uma única medição e duas linhas. Se `PROTOCOLO` ou
`SOLICDATA` entrassem na chave, as duas linhas pareceriam registros distintos e a
duplicidade passaria despercebida — que é exatamente o erro a evitar.

### Classificação

| Classificação | Significado | Ação |
|---|---|---|
| `REENVIO` | Mesma leitura lançada mais de uma vez, com valores iguais | Deduplicação segura: manter uma linha |
| `CONFLITO DE VALOR` | Mesma chave com valores diferentes | Exige critério explícito de qual prevalece |

As colunas `protocolos` e `datas_lancamento` no resultado mostram quais lançamentos
originaram cada duplicidade.

In [0]:
# Pluviômetros — chave (estação, data de registro)

dup_pluv = resumo_duplicidade(bronze_pluv, ["ESTACAO", "DATAREGISTRO"], ["MILIMETROS"],"leitura_pluviometros")
display(dup_pluv)

In [0]:
# Bacias — chave (bacia, data de registro)

dup_bacia = resumo_duplicidade(bronze_bacia, ["BACIA", "DATAREGISTRO"],["AGUADOCE_COLOCADO", "SALMOURA_COLOCADO", "SALMOURA_DENSIDADE"],"medicao_bacias")
display(dup_bacia)

In [0]:
# Laje — chave (cristalizador, data de medição)

dup_laje = resumo_duplicidade(bronze_laje, ["CRISTALIZADOR", "DATAMEDICAO"],
                              ["REGUA_1", "REGUA_2", "REGUA_3", "REGUA_4","REGUA_5", "REGUA_6", "REGUA_7", "REGUA_8"],
                              "medicao_laje")
display(dup_laje)

### Detalhamento das linhas duplicadas

Exibe as linhas completas por trás de cada chave duplicada, lado a lado, para que a
decisão de qual registro manter seja tomada com o dado à vista.

In [0]:
chaves_dup_laje = dup_laje.select("CRISTALIZADOR", "DATAMEDICAO")

display(bronze_laje.join(chaves_dup_laje, ["CRISTALIZADOR", "DATAMEDICAO"])
        .select("CRISTALIZADOR", "DATAMEDICAO", "PROTOCOLO", "SOLICDATA",
                *[f"REGUA_{i}" for i in range(1, 9)])
        .orderBy("CRISTALIZADOR", "DATAMEDICAO", "SOLICDATA"))

In [0]:
chaves_dup_pluv = dup_pluv.select("ESTACAO", "DATAREGISTRO")

display(bronze_pluv.join(chaves_dup_pluv, ["ESTACAO", "DATAREGISTRO"])
        .select("ESTACAO", "DATAREGISTRO", "PROTOCOLO", "SOLICDATA", "MILIMETROS")
        .orderBy("DATAREGISTRO", "ESTACAO", "SOLICDATA"))

In [0]:
chaves_dup_bacia = dup_bacia.select("BACIA", "DATAREGISTRO")

display(bronze_bacia.join(chaves_dup_bacia, ["BACIA", "DATAREGISTRO"])
        .select("BACIA", "DATAREGISTRO", "PROTOCOLO", "SOLICDATA",
                "AGUADOCE_COLOCADO", "SALMOURA_COLOCADO", "SALMOURA_DENSIDADE")
        .orderBy("DATAREGISTRO", "BACIA", "SOLICDATA"))

In [0]:
# A medição é diária. Dias faltantes em uma bacia e presentes na outra reforçam o diagnóstico de registro faltante.

dias_por_bacia = (bronze_bacia
    .groupBy("BACIA")
    .agg(F.countDistinct("DATAREGISTRO").alias("dias_com_registro"),
         F.min("DATAREGISTRO").alias("primeira_data"),
         F.max("DATAREGISTRO").alias("ultima_data")))

display(dias_por_bacia)

datas_b1 = {r[0] for r in bronze_bacia.filter(F.col("BACIA") == "BACIA1").select("DATAREGISTRO").distinct().collect()}
datas_b2 = {r[0] for r in bronze_bacia.filter(F.col("BACIA") == "BACIA2").select("DATAREGISTRO").distinct().collect()}

print(f"Datas presentes na BACIA1 e ausentes na BACIA2: {sorted(datas_b1 - datas_b2)}")
print(f"Datas presentes na BACIA2 e ausentes na BACIA1: {sorted(datas_b2 - datas_b1)}")

## 2.2 Acurácia — faixa de valores

Verifica se os valores numéricos fazem sentido no contexto físico do processo.

In [0]:
# Verificação de reguas com leitura máxima

laje_num = bronze_laje.select(
    texto_para_numero("CRISTALIZADOR", "int").alias("cristalizador"),
    F.col("DATAMEDICAO").alias("data_medicao"),
    *[texto_para_numero(c, "int").alias(c.lower()) for c in reguas])

cols_r = [F.col(c.lower()) for c in reguas]

display(laje_num
    .withColumn("reguas_em_100", sum([F.when(c == 100, 1).otherwise(0) for c in cols_r]))
    .filter(F.col("reguas_em_100") > 0)
    .orderBy(F.desc("reguas_em_100")))

In [0]:
# Volumes retirados muito acima da faixa usual correspondem a eventos de chuva forte. São valores legítimos, mas precisam ser identificados para não serem descartados em etapas futuras

display(bronze_bacia
    .select("BACIA", "DATAREGISTRO",
            texto_para_numero("AGUADOCE_RETIRADO", "int").alias("agua_retirado"),
            texto_para_numero("SALMOURA_RETIRADO", "int").alias("salmoura_retirado"),
            texto_para_numero("SALMOURA_DENSIDADE").alias("densidade_be"))
    .filter((F.col("agua_retirado") > 15) | (F.col("salmoura_retirado") > 15))
    .orderBy(F.desc("agua_retirado")))

## 2.3 Completude — nulos após a conversão de tipos

Mede quanto de cada coluna estará efetivamente preenchido na Silver.

In [0]:
display(perfil_completude(perfil, ["agua_colocado", "agua_retirado", "salm_colocado",
                                   "salm_retirado", "densidade_be"]))

In [0]:
display(bronze_bacia
    .groupBy("BACIA")
    .agg(F.count("*").alias("registros"),
         F.sum(F.when(texto_para_numero("SALMOURA_DENSIDADE").isNotNull(), 1).otherwise(0))
          .alias("com_densidade"))
    .withColumn("pct_com_densidade",
                F.round(100.0 * F.col("com_densidade") / F.col("registros"), 1)))

## 🥈 Etapa 3: Transformação Bronze → Silver

Lê cada tabela da camada **Bronze**, aplica limpeza e padronização, e grava na camada **Silver**.

### Transformações aplicadas:
* **Normalização de nomes**: colunas em lowercase com `_` (ex: `DATAREGISTRO` → `data_registro`)
* **Conversão de datas**: strings `dd/MM/yyyy` → `DATE` e `dd/MM/yyyy HH:mm:ss` → `TIMESTAMP`
* **Conversão de decimais**: vírgula → ponto (ex: `14,0` → `14.0` → `DOUBLE`)
* **Tratamento de valores ausentes**: `-` → `NULL`
* **Metadados de rastreabilidade**: `data_carga_silver` e `origem_tabela_bronze`
* **Aplicado filtro**: `STATUS = Solicitação Finalizada`
* **Campos removidos**: `PROTOCOLO`, `SOLICDATA`, `STATUS`, `UNIDADE`, `OBSERVACOES` por não trazer informações significativas para analise.

O campo `DATAREGISTRO` é o momento que a leitura foi realizada em campo e este é importante para as análises.

### Princípios da camada Silver:
* ✅ **Dados limpos** - tipos corretos, sem valores inválidos
* ✅ **Nomes padronizados** - lowercase com snake_case
* ✅ **Preserva rastreabilidade** - mantém metadados de staging e bronze
* ✅ **Formato Delta Lake** - formato otimizado do Databricks

In [0]:
# =========================================
# TRANSFORMAÇÃO: leitura_pluviometros
# =========================================
# Transformações aplicadas:
#   - DATAREGISTRO: STRING "dd/MM/yyyy" para DATE
#   - MILIMETROS: vírgula decimal para ponto e STRING para DOUBLE
#   - Nomes de colunas: uppercase para lowercase com _
#   - Aplicado filtro para STATUS = Solicitação Finalizada
#   - Removido campos: OBSERVACAO, SOLICDATA, STATUS, UNIDADE E OBSERVACOES. 

pluv = (bronze_pluv
    .filter(F.col("STATUS") == STATUS_HOMOLOGADO)
    .select(
        F.regexp_replace(F.trim(F.col("ESTACAO")), r"\.$", "").alias("estacao"), #retira o ponto do nome da estação
        F.to_date(F.col("DATAREGISTRO"), "dd/MM/yyyy").alias("data_registro"),
        texto_para_numero("MILIMETROS").alias("milimetros"),
        F.to_timestamp(F.col("SOLICDATA"), "dd/MM/yyyy HH:mm:ss").alias("_solicdata"),
        F.col("data_ingestao"), F.col("origem"), F.col("arquivo_origem"),
        F.col("data_carga_bronze")))

# Deduplicação: preenchido vence nulo; depois, lançamento mais recente
janela = (Window.partitionBy("estacao", "data_registro")
          .orderBy(F.col("milimetros").isNotNull().desc(),
                   F.col("_solicdata").desc_nulls_last(),
                   F.col("milimetros").desc_nulls_last()))

antes = pluv.count()

silver_pluv = (pluv
    .withColumn("_rn", F.row_number().over(janela))
    .filter(F.col("_rn") == 1)
    .drop("_rn", "_solicdata")
    .withColumn("data_carga_silver", F.current_timestamp()))

depois = silver_pluv.count()

(silver_pluv.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOGO}.silver.leitura_pluviometros"))

print(f"Registros na bronze ............: {bronze_pluv.count()}")
print(f"Após filtro de status ..........: {antes}")
print(f"Após deduplicação ..............: {depois}   (removidos: {antes - depois})")
print(f"Estações distintas .............: {silver_pluv.select('estacao').distinct().count()}")

In [0]:
# =========================================
# TRANSFORMAÇÃO: medicao_bacias
# =========================================
# Transformações aplicadas:
#   - SOLICDATA de STRING "dd/MM/yyyy HH:mm:ss" para TIMESTAMP
#   - DATAREGISTRO de STRING "dd/MM/yyyy" para DATE
#   - AGUADOCE_COLOCADO, AGUADOCE_RETIRADO, SALMOURA_COLOCADO e SALMOURA_RETIRADO transformado "-" em NULL, depois STRING  para INT
#   - SALMOURA_DENSIDADE: vírgula decimal para ponto e STRING para DOUBLE
#   - Nomes de colunas: uppercase para lowercase com _
#   - Removido campo de observeção
#   - Aplicado filtro para STATUS = Solicitação Finalizada
# =========================================
# Nota sobre as duas correções

# O diagnóstico da etapa 2 identificou dois protocolos que violam a regra de emparelhamento, e o cruzamento com a cobertura da série diária permite determinar a correção de cada um:

# **Protocolo 283327.** A linha registrada como `BACIA1 / 15-01` contém apenas salmoura, sem água doce e sem densidade — perfil da BACIA2, não da BACIA1. A BACIA1 já tem registro completo nessa data pelo protocolo 283326, e a BACIA2 não tem registro algum em 15-01. O registro é da BACIA2 e foi rotulado como BACIA1.

# **Protocolo 326238.** A linha da BACIA2 está datada 12-06, mas o par BACIA1 do mesmo protocolo é de 11-06, e a BACIA2 não tem registro em 11-06. A data está errada.

# As correções são aplicadas explicitamente, com o protocolo como chave, e ficam registradas no código. Não são deduplicação silenciosa.

bacia_corrigida = (bronze_bacia
    .filter(F.col("STATUS") == STATUS_HOMOLOGADO)
    # Correção 1: registro da BACIA2 rotulado como BACIA1
    .withColumn("BACIA",
        F.when((F.col("PROTOCOLO") == "283327") & (F.col("DATAREGISTRO") == "15/01/2026"),
               F.lit("BACIA2")).otherwise(F.col("BACIA")))
    # Correção 2: data errada no registro da BACIA2
    .withColumn("DATAREGISTRO",
        F.when((F.col("PROTOCOLO") == "326238") & (F.col("BACIA") == "BACIA2")
               & (F.col("DATAREGISTRO") == "12/06/2026"),
               F.lit("11/06/2026")).otherwise(F.col("DATAREGISTRO"))))

bacia = bacia_corrigida.select(
    F.col("BACIA").alias("bacia"),
    F.to_date(F.col("DATAREGISTRO"), "dd/MM/yyyy").alias("data_registro"),
    texto_para_numero("AGUADOCE_COLOCADO", "int").alias("agua_doce_colocado"),
    texto_para_numero("AGUADOCE_RETIRADO", "int").alias("agua_doce_retirado"),
    texto_para_numero("SALMOURA_COLOCADO", "int").alias("salmoura_colocado"),
    texto_para_numero("SALMOURA_RETIRADO", "int").alias("salmoura_retirado"),
    texto_para_numero("SALMOURA_DENSIDADE").alias("salmoura_densidade"),
    F.to_timestamp(F.col("SOLICDATA"), "dd/MM/yyyy HH:mm:ss").alias("_solicdata"),
    F.col("data_ingestao"), F.col("origem"), F.col("arquivo_origem"),
    F.col("data_carga_bronze"))

janela_b = (Window.partitionBy("bacia", "data_registro")
            .orderBy(F.col("salmoura_densidade").isNotNull().desc(),
                     F.col("_solicdata").desc_nulls_last()))

antes = bacia.count()

silver_bacia = (bacia
    .withColumn("_rn", F.row_number().over(janela_b))
    .filter(F.col("_rn") == 1)
    .drop("_rn", "_solicdata")
    .withColumn("data_carga_silver", F.current_timestamp()))

depois = silver_bacia.count()

(silver_bacia.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOGO}.silver.medicao_bacias"))

print(f"Registros na bronze ............: {bronze_bacia.count()}")
print(f"Após filtro de status ..........: {antes}   (removidos: {bronze_bacia.count() - antes})")
print(f"Após correções e deduplicação ..: {depois}")

In [0]:
# =========================================
# TRANSFORMAÇÃO: medicao_laje
# =========================================
# Transformações aplicadas:
#   - SOLICDATA de STRING "dd/MM/yyyy HH:mm:ss" para TIMESTAMP
#   - DATAMEDICAO: string "dd/MM/yyyy" para DATE
#   - REGUA_1 a REGUA_8 para INT
#   - AREAHECTARES para DOUBLE
#   - Nomes de colunas: uppercase para lowercase com _
#   - Removido campo de observeção
#   - Aplicado filtro para STATUS = Solicitação Finalizada

laje = (bronze_laje
    .filter(F.col("STATUS") == STATUS_HOMOLOGADO)
    .select(
        F.to_date(F.col("DATAMEDICAO"), "dd/MM/yyyy").alias("data_medicao"),
        F.col("CRISTALIZADOR").alias("cristalizador"),
        texto_para_numero("AREAHECTARES").alias("area_hectares"),
        *[texto_para_numero(c, "int").alias(c.lower()) for c in reguas],
        F.to_timestamp(F.col("SOLICDATA"), "dd/MM/yyyy HH:mm:ss").alias("_solicdata"),
        F.col("data_ingestao"), F.col("origem"), F.col("arquivo_origem"),
        F.col("data_carga_bronze")))

colunas_regua = [F.col(f"regua_{i}") for i in range(1, 9)]

janela_l = (Window.partitionBy("cristalizador", "data_medicao")
            .orderBy(F.col("_solicdata").desc_nulls_last()))

antes = laje.count()

silver_laje = (laje
    .withColumn("_rn", F.row_number().over(janela_l))
    .filter(F.col("_rn") == 1)
    .drop("_rn", "_solicdata")
    .withColumn("media_reguas_cm",
                F.round(sum(colunas_regua) / F.lit(8), 3))
    .withColumn("amplitude_reguas_cm",
                F.greatest(*colunas_regua) - F.least(*colunas_regua))
    .withColumn("flag_censura_regua",
                F.least(*colunas_regua) >= 100)
    .withColumn("data_carga_silver", F.current_timestamp()))

depois = silver_laje.count()

(silver_laje.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOGO}.silver.medicao_laje"))

print(f"Registros na bronze ............: {bronze_laje.count()}")
print(f"Após filtro de status ..........: {antes}")
print(f"Após deduplicação ..............: {depois}   (removidos: {antes - depois})")
print(f"Medições com régua censurada ...: {silver_laje.filter('flag_censura_regua').count()}")

## 🔍 Etapa 4: Validação dos Dados Silver

Os mesmos testes da etapa 2, agora sobre a Silver. Todos devem retornar vazio ou o resultado esperado.

In [0]:
s_pluv  = spark.table(f"{CATALOGO}.silver.leitura_pluviometros")
s_bacia = spark.table(f"{CATALOGO}.silver.medicao_bacias")
s_laje  = spark.table(f"{CATALOGO}.silver.medicao_laje")

falhas = []

# 4.1 Unicidade
for nome, df, chaves in [("leitura_pluviometros", s_pluv,  ["estacao", "data_registro"]),
                         ("medicao_bacias",       s_bacia, ["bacia", "data_registro"]),
                         ("medicao_laje",         s_laje,  ["cristalizador", "data_medicao"])]:
    dup = df.groupBy(*chaves).count().filter(F.col("count") > 1).count()
    ok = dup == 0
    print(f"{'✓' if ok else '✗'} Unicidade de {nome}: {dup} chave(s) duplicada(s)")
    if not ok:
        falhas.append(f"{nome}: {dup} duplicatas")

# 4.2 Estações normalizadas
estacoes = sorted(r[0] for r in s_pluv.select("estacao").distinct().collect())
ok = len(estacoes) == 5
print(f"{'✓' if ok else '✗'} Estações distintas: {len(estacoes)} → {estacoes}")
if not ok:
    falhas.append(f"estações: {len(estacoes)} em vez de 5")

# 4.3 Domínios físicos
checagens = [
    ("densidade fora de 20–30 °Bé",
     s_bacia.filter(F.col("salmoura_densidade").isNotNull()
                    & ~F.col("salmoura_densidade").between(20, 30)).count()),
    ("média de réguas fora de 50–100 cm",
     s_laje.filter(~F.col("media_reguas_cm").between(50, 100)).count()),
    ("precipitação fora de 0–200 mm",
     s_pluv.filter(F.col("milimetros").isNotNull()
                   & ~F.col("milimetros").between(0, 200)).count()),
    ("cristalizador fora de 1–25",
     s_laje.filter(~F.col("cristalizador").between(1, 25)).count()),
]
for descricao, qtd in checagens:
    ok = qtd == 0
    print(f"{'✓' if ok else '✗'} {descricao}: {qtd} registro(s)")
    if not ok:
        falhas.append(f"{descricao}: {qtd}")

# 4.4 Colunas descartadas não devem existir
descartadas = {"protocolo", "solicdata", "status", "unidade", "observacoes"}
for nome, df in [("leitura_pluviometros", s_pluv), ("medicao_bacias", s_bacia), ("medicao_laje", s_laje)]:
    residuais = descartadas & {c.lower() for c in df.columns}
    ok = not residuais
    print(f"{'✓' if ok else '✗'} Colunas descartadas ausentes em {nome}: {sorted(residuais) or 'nenhuma residual'}")
    if not ok:
        falhas.append(f"{nome}: colunas residuais {residuais}")

# 4.5 Datas convertidas
for nome, df, col in [("leitura_pluviometros", s_pluv, "data_registro"),
                      ("medicao_bacias", s_bacia, "data_registro"),
                      ("medicao_laje", s_laje, "data_medicao")]:
    nulos = df.filter(F.col(col).isNull()).count()
    ok = nulos == 0
    print(f"{'✓' if ok else '✗'} Datas convertidas em {nome}: {nulos} nulo(s)")
    if not ok:
        falhas.append(f"{nome}: {nulos} datas nulas")

print("\n" + "=" * 70)
if falhas:
    raise RuntimeError(f"Validação da Silver falhou: {falhas}")
print("Todas as validações de qualidade passaram.")

In [0]:
# Confirmação das correções aplicadas nas bacias
# As duas datas corrigidas devem agora aparecer nas duas bacias.

display(s_bacia.filter(F.col("data_registro").isin("2026-01-15", "2026-06-11", "2026-06-12")).orderBy("data_registro", "bacia"))

In [0]:
# Cobertura final da série

display(s_bacia.groupBy("bacia").agg(
    F.count("*").alias("registros"),
    F.countDistinct("data_registro").alias("dias_distintos"),
    F.min("data_registro").alias("primeira_data"),
    F.max("data_registro").alias("ultima_data"),
    F.sum(F.when(F.col("salmoura_densidade").isNotNull(), 1).otherwise(0)).alias("com_densidade")))

## 📝 Etapa 5: Documentação das Tabelas e Colunas

Comentários no Unity Catalog tornam os dados autodescritivos, visíveis no Catalog Explorer, autocomplete do SQL Editor e APIs.

In [0]:
%sql
-- =========================================
-- COMENTÁRIOS DA TABELA: leitura_pluviometros
-- =========================================

COMMENT ON TABLE mvpdataengineer.silver.leitura_pluviometros IS
  'Camada Silver. Leituras pluviométricas limpas, tipadas e deduplicadas, uma linha por estação e data. Rótulos das estações normalizados. Série registrada por evento: a ausência de linha não equivale a ausência de chuva.';

COMMENT ON COLUMN mvpdataengineer.silver.leitura_pluviometros.estacao IS 'Estação pluviométrica, rótulo normalizado. Domínio: PLUVIOMETRO1 a PLUVIOMETRO5.';
COMMENT ON COLUMN mvpdataengineer.silver.leitura_pluviometros.data_registro IS 'Data da leitura em campo (DATE).';
COMMENT ON COLUMN mvpdataengineer.silver.leitura_pluviometros.milimetros IS 'Precipitação medida em milímetros (DOUBLE). Domínio esperado: 0 a 200. Nulo quando a leitura não foi registrada.';
COMMENT ON COLUMN mvpdataengineer.silver.leitura_pluviometros.data_carga_silver IS 'Timestamp da carga na camada Silver.';

In [0]:
%sql
-- =========================================
-- COMENTÁRIOS DA TABELA: medicao_bacias
-- =========================================

COMMENT ON TABLE mvpdataengineer.silver.medicao_bacias IS
  'Camada Silver. Medição diária de evaporação nas bacias, limpa, tipada e deduplicada, uma linha por bacia e dia. Inclui duas correções de registro atribuído à bacia ou à data errada, documentadas no notebook MVP04.';

COMMENT ON COLUMN mvpdataengineer.silver.medicao_bacias.bacia IS 'Ponto de medição de evaporação.';
COMMENT ON COLUMN mvpdataengineer.silver.medicao_bacias.data_registro IS 'Data da medição em campo (DATE).';
COMMENT ON COLUMN mvpdataengineer.silver.medicao_bacias.agua_doce_colocado IS 'Volume de água doce reposto até a marca, em litros (INT). Corresponde à evaporação do dia. Mutuamente exclusivo com agua_doce_retirado.';
COMMENT ON COLUMN mvpdataengineer.silver.medicao_bacias.agua_doce_retirado IS 'Volume de água doce retirado por excedente de chuva, em litros (INT). Mutuamente exclusivo com agua_doce_colocado.';
COMMENT ON COLUMN mvpdataengineer.silver.medicao_bacias.salmoura_colocado IS 'Volume de salmoura reposto até a marca, em litros (INT). Corresponde à evaporação do dia. Mutuamente exclusivo com salmoura_retirado.';
COMMENT ON COLUMN mvpdataengineer.silver.medicao_bacias.salmoura_retirado IS 'Volume de salmoura retirado por excedente de chuva, em litros (INT). Mutuamente exclusivo com salmoura_colocado.';
COMMENT ON COLUMN mvpdataengineer.silver.medicao_bacias.salmoura_densidade IS 'Concentração da salmoura em graus Baumé, °Bé (DOUBLE). Domínio observado: 22,0 a 28,0. Registrada apenas em um dos pontos de medição.';
COMMENT ON COLUMN mvpdataengineer.silver.medicao_bacias.data_carga_silver IS 'Timestamp da carga na camada Silver.';

In [0]:
%sql
-- =========================================
-- COMENTÁRIOS DA TABELA: medicao_laje
-- =========================================

COMMENT ON TABLE mvpdataengineer.silver.medicao_laje IS
  'Camada Silver. Medição mensal da laje dos cristalizadores, limpa, tipada e deduplicada, uma linha por cristalizador e data. Inclui a média e a amplitude das 8 réguas e a marcação de censura no limite do instrumento.';

COMMENT ON COLUMN mvpdataengineer.silver.medicao_laje.cristalizador IS 'Identificador do cristalizador (INT). Domínio: 1 a 25.';
COMMENT ON COLUMN mvpdataengineer.silver.medicao_laje.data_medicao IS 'Data da medição em campo (DATE). O intervalo entre medições consecutivas varia de 25 a 35 dias.';
COMMENT ON COLUMN mvpdataengineer.silver.medicao_laje.area_hectares IS 'Área do cristalizador em hectares (DOUBLE). Constante por cristalizador: atributo de dimensão.';
COMMENT ON COLUMN mvpdataengineer.silver.medicao_laje.media_reguas_cm IS 'Média das 8 réguas, em cm (DOUBLE). Medida representativa da altura livre acima da laje. Domínio esperado: 50 a 100.';
COMMENT ON COLUMN mvpdataengineer.silver.medicao_laje.amplitude_reguas_cm IS 'Diferença entre a maior e a menor régua da mesma medição, em cm (INT). Indicador de uniformidade da deposição.';
COMMENT ON COLUMN mvpdataengineer.silver.medicao_laje.flag_censura_regua IS 'Verdadeiro quando todas as réguas atingiram 100 cm, limite físico do instrumento. Nesses casos a altura livre real é maior ou igual ao valor registrado.';
COMMENT ON COLUMN mvpdataengineer.silver.medicao_laje.data_carga_silver IS 'Timestamp da carga na camada Silver.';

In [0]:
resumo_qualidade = [
    {"dimensao": "Consistência", "tabela": "leitura_pluviometros",
     "problema": "PLUVIOMETRO4. e PLUVIOMETRO4 como rótulos distintos da mesma estação",
     "tratamento": "Remoção do ponto final; dimensão passa de 6 para 5 membros"},
    {"dimensao": "Consistência", "tabela": "todas",
     "problema": "Ausência representada ora por hífen, ora por célula vazia",
     "tratamento": "Ambas convertidas para NULL na tipagem"},
    {"dimensao": "Consistência", "tabela": "todas",
     "problema": "Decimal com vírgula impedindo conversão numérica",
     "tratamento": "Substituição por ponto antes do cast"},
    {"dimensao": "Unicidade", "tabela": "medicao_laje",
     "problema": "Mesma medição (cristalizador, data) lançada em mais de um protocolo",
     "tratamento": "Deduplicação pela chave de negócio, mantendo o lançamento mais recente por SOLICDATA"},
    {"dimensao": "Unicidade", "tabela": "leitura_pluviometros",
     "problema": "Mesma leitura (estação, data) lançada em mais de um protocolo, com valores divergentes",
     "tratamento": "Registro preenchido prevalece sobre nulo; empate resolvido pelo lançamento mais recente"},
    {"dimensao": "Unicidade", "tabela": "todas",
     "problema": "PROTOCOLO e SOLICDATA identificam o lançamento, não a medição",
     "tratamento": "Mantidos fora da chave de duplicidade; usados apenas como critério de desempate"},
    {"dimensao": "Acurácia", "tabela": "medicao_bacias",
     "problema": "Registro da BACIA2 rotulado como BACIA1 (protocolo 283327)",
     "tratamento": "Reclassificação explícita, detectada pela regra de emparelhamento de protocolo"},
    {"dimensao": "Acurácia", "tabela": "medicao_bacias",
     "problema": "Registro da BACIA2 com data incorreta (protocolo 326238)",
     "tratamento": "Correção da data para 11/06/2026"},
    {"dimensao": "Acurácia", "tabela": "medicao_laje",
     "problema": "Censura no limite de 100 cm da régua em medições pós-colheita",
     "tratamento": "Marcação com flag_censura_regua; valor real é maior ou igual ao registrado"},
    {"dimensao": "Completude", "tabela": "todas",
     "problema": "Registros não homologados no fluxo de aprovação",
     "tratamento": "Filtro por STATUS = 'Solicitação Finalizada' antes do descarte da coluna"},
    {"dimensao": "Completude", "tabela": "medicao_bacias",
     "problema": "Densidade ausente em um dos pontos de medição",
     "tratamento": "Mantido na Silver; o recorte de escopo é aplicado na camada Gold"},
    {"dimensao": "Relevância", "tabela": "todas",
     "problema": "Colunas sem valor analítico (protocolo, solicdata, status, unidade, observações)",
     "tratamento": "Descartadas após cumprirem sua função de filtro e desempate"},
]

display(spark.createDataFrame(resumo_qualidade))